In [51]:
##INSTALAR DEPENDENCIAS
!pip install requests beautifulsoup4

In [52]:
##IMPORTANCIONES Y CONEXCION A MONGO
import requests
from bs4 import BeautifulSoup
import time
import random
from pymongo import MongoClient
from datetime import datetime
from dotenv import load_dotenv
import os
import json

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["proyecto2_musica"]
coleccion = db["canciones"]

print("Xonectado a MongoDB Atlas!")

Xonectado a MongoDB Atlas!


In [53]:
##VERIFICAR ROBOTS
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

resp = requests.get("https://www.lyrics.com/robots.txt", headers=HEADERS)
print("robots.txt de lyrics.com:")
print(resp.text)

robots.txt de lyrics.com:



In [54]:
##BUENAS PRACTICAS
CONFIG = {
    "delay_min": 4,   # 2
    "delay_max": 7,   # 4
    "max_reintentos": 3,
    "timeout": 10,
    "max_errores": 10,
}

errores_consecutivos = 0

def request_seguro(url):
    global errores_consecutivos

    for intento in range(CONFIG["max_reintentos"]):
        try:
            time.sleep(random.uniform(CONFIG["delay_min"], CONFIG["delay_max"]))
            resp = requests.get(url, headers=HEADERS, timeout=CONFIG["timeout"])

            if resp.status_code == 200:
                errores_consecutivos = 0
                return resp
            elif resp.status_code == 429:
                print(f"Rate limit, esperando 30 segundos...")
                time.sleep(30)
            else:
                print(f"Status {resp.status_code} en intento {intento+1}")

        except Exception as e:
            print(f"Error intento {intento+1}: {e}")
            time.sleep(5)

    errores_consecutivos += 1
    if errores_consecutivos >= CONFIG["max_errores"]:
        raise Exception("Demasiados errores, scraping detenido!")

    return None

print("Buenas prácticas configuradas")
print(f"Delay: {CONFIG['delay_min']}-{CONFIG['delay_max']} segundos")
print(f"Reintentos: {CONFIG['max_reintentos']}")
print(f"Para tras: {CONFIG['max_errores']} errores seguidos")

Buenas prácticas configuradas
Delay: 4-7 segundos
Reintentos: 3
Para tras: 10 errores seguidos


In [55]:
##FUNCION PRINCIPAL SCRAPING
def get_urls_artista(artista_slug):
    url = f"https://www.lyrics.com/artist/{artista_slug}/"
    resp = request_seguro(url)
    if not resp:
        return []

    soup = BeautifulSoup(resp.text, "html.parser")
    links = soup.find_all("a", href=True)

    urls = []
    for link in links:
        href = link["href"]
        if "/lyric/" in href and href != "/lyric/0":
            if not href.startswith("http"):
                href = f"https://www.lyrics.com{href}"
            urls.append(href)

    return list(set(urls))


def scrape_letra(url_cancion):
    try:
        resp = request_seguro(url_cancion)
        if not resp:
            return None

        soup = BeautifulSoup(resp.text, "html.parser")

        # Extraer letra
        pre = soup.find("pre", id="lyric-body-text")
        if not pre:
            return None
        letra = pre.get_text().strip()

        # Extraer título y artista
        titulo = soup.find("h1", class_="lyric-title")
        artista = soup.find("h3", class_="lyric-artist")

        if not titulo:
            titulo = soup.find("h1")
        if not artista:
            artista = soup.find("h3")

        return {
            "titulo": titulo.text.strip() if titulo else "Desconocido",
            "artista": artista.text.strip() if artista else "Desconocido",
            "letra": letra,
            "fuente": "scraping",
            "url_fuente": url_cancion,
            "fecha_recopilacion": datetime.now(),
            "genero": None,
            "idioma": "es",
            "embeddings": {"word2vec_avg": None, "beto_cls": None}
        }
    except Exception as e:
        print(f"Error: {e}")
        return None

In [56]:
##OBTENER URLS DE LAS CANCIONES POR ARTISTAS
artistas = [
    "Shakira", "Bad-Bunny", "pxndx", "Jose-Madero",
    "Daddy-Yankee", "Marc-Anthony", "Carlos-Vives", "Juanes",
    "Alejandro-Sanz", "Ricky-Martin", "Enrique-Iglesias",
    "Nicky-Jam", "Karol-G", "Aventura", "Romeo-Santos",
    "Maná", "Wisin-Yandel", "Don-Omar", "Pitbull"
]

print(f" {len(artistas)} artistas listos")

 19 artistas listos


In [57]:
##GUARDAR Y SEGUIR
CHECKPOINT_FILE = "../data/processed/scraping_checkpoint.json"

def guardar_checkpoint(artista_actual, urls_procesadas, total_insertadas):
    checkpoint = {
        "artista_actual": artista_actual,
        "urls_procesadas": list(urls_procesadas),
        "total_insertadas": total_insertadas,
        "ultima_actualizacion": datetime.now().isoformat()
    }
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(checkpoint, f)

def cargar_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            contenido = f.read().strip()
            if not contenido:
                return None
            checkpoint = json.loads(contenido)
        print(f" Checkpoint encontrado!")
        print(f" Último artista: {checkpoint['artista_actual']}")
        print(f" URLs procesadas: {len(checkpoint['urls_procesadas'])}")
        print(f" Canciones insertadas: {checkpoint['total_insertadas']}")
        return checkpoint
    else:
        print("No hay checkpoint, empezando desde cero")
        return None

checkpoint = cargar_checkpoint()
if checkpoint:
    urls_procesadas = set(checkpoint["urls_procesadas"])
    total_insertadas = checkpoint["total_insertadas"]
    artista_inicio = checkpoint["artista_actual"]
    if artista_inicio in artistas:
        idx = artistas.index(artista_inicio)
        artistas = artistas[idx:]
    print(f"Retomando desde: {artista_inicio}")
else:
    urls_procesadas = set()
    total_insertadas = 0

 Checkpoint encontrado!
 Último artista: j-balvin
 URLs procesadas: 34
 Canciones insertadas: 0
Retomando desde: j-balvin


In [58]:
##SCRAPING MASIVO Y GUARDAR EN MONGO
META = 1000
total_errores = 0

#contador de canciones insertadas en mongodb
total_insertadas = coleccion.count_documents({"fuente": "scraping"})
print(f" Contador : {total_insertadas} canciones ya scrapeadas")

try:
    for artista in artistas:
        if total_insertadas >= META:
            break

        print(f"\n Scrapeando: {artista}")
        urls = get_urls_artista(artista)
        print(f"   URLs encontradas: {len(urls)}")

        for url in urls:
            if total_insertadas >= META:
                break

            if url in urls_procesadas:
                continue

            if coleccion.find_one({"url_fuente": url}):
                urls_procesadas.add(url)
                continue

            cancion = scrape_letra(url)
            urls_procesadas.add(url)

            if cancion:
                coleccion.insert_one(cancion)
                total_insertadas += 1
                print(f" [{total_insertadas}/{META}] {cancion['titulo']} - {cancion['artista']}")
            else:
                total_errores += 1

            if total_insertadas % 50 == 0 and total_insertadas > 0:
                guardar_checkpoint(artista, urls_procesadas, total_insertadas)
                print(f" Checkpoint guardado en {total_insertadas} canciones")

except Exception as e:
    print(f"\n Scraping detenido: {e}")
    guardar_checkpoint(artista, urls_procesadas, total_insertadas)
    print(f" Checkpoint guardado. Podés retomar desde donde quedó.")

finally:
    print(f"\n Sesión finalizada!")
    print(f" Canciones scrapeadas en total: {total_insertadas}")
    print(f" Errores: {total_errores}")
    print(f" Total en MongoDB: {coleccion.count_documents({})}")

 Contador corregido: 24 canciones ya scrapeadas

🎵 Scrapeando: Shakira
Status 202 en intento 1
Status 202 en intento 2
Status 202 en intento 3
   URLs encontradas: 0

🎵 Scrapeando: Bad-Bunny
Status 202 en intento 1
Status 202 en intento 2
Status 202 en intento 3
   URLs encontradas: 0

🎵 Scrapeando: pxndx
Status 202 en intento 1
Status 202 en intento 2
Status 202 en intento 3
   URLs encontradas: 0

🎵 Scrapeando: Jose-Madero
Status 202 en intento 1
Status 202 en intento 2
Status 202 en intento 3
   URLs encontradas: 0

🎵 Scrapeando: Daddy-Yankee
Status 202 en intento 1
Status 202 en intento 2
Status 202 en intento 3
   URLs encontradas: 0

🎵 Scrapeando: Marc-Anthony
Status 202 en intento 1
Status 202 en intento 2
Status 202 en intento 3
   URLs encontradas: 0

🎵 Scrapeando: Carlos-Vives
Status 202 en intento 1
Status 202 en intento 2
Status 202 en intento 3
   URLs encontradas: 0

🎵 Scrapeando: Juanes
Status 202 en intento 1
Status 202 en intento 2
Status 202 en intento 3
   URLs encon

In [59]:
##CERRAR CONEXION
client.close()
print("Conexión cerrada")

Conexión cerrada


In [61]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["proyecto2_musica"]
coleccion = db["canciones"]

# Total general
print(f" Total en MongoDB: {coleccion.count_documents({})}")

# Por fuente
print(f"\n Por fuente:")
for doc in coleccion.aggregate([{"$group": {"_id": "$fuente", "total": {"$sum": 1}}}]):
    print(f"  {doc['_id']}: {doc['total']}")

# Por género
print(f"\n🎵 Por género:")
for doc in coleccion.aggregate([{"$group": {"_id": "$genero", "total": {"$sum": 1}}}]):
    print(f"  {doc['_id']}: {doc['total']}")

# Ejemplo de canción scrapeada
print(f"\n Ejemplo scrapeado:")
ej = coleccion.find_one({"fuente": "scraping"})
if ej:
    print(f"  Título: {ej['titulo']}")
    print(f"  Artista: {ej['artista']}")
    print(f"  Letra (primeras 100 letras): {ej['letra'][:100]}")

📊 Total en MongoDB: 10024

 Por fuente:
  kaggle: 10000
  scraping: 24

🎵 Por género:
  jazz: 1328
  country: 1950
  None: 24
  hip hop: 321
  rock: 1404
  pop: 2494
  blues: 1610
  reggae: 893

 Ejemplo scrapeado:
  Título: Ready for the Good Times
  Artista: Shakira
  Letra (primeras 100 letras): I don't wanna clear the cobwebs from my head
Time will bring them back, I bet
So, if you hear me say


In [60]:
##comprobar urls
resp = requests.get("https://genius.com/Shakira-waka-waka-lyrics", headers=HEADERS)
print(resp.status_code)
soup = BeautifulSoup(resp.text, "html.parser")

# Buscar la letra
for div in soup.find_all("div", attrs={"data-lyrics-container": "true"}):
    print(div.get_text()[:300])
    print("---")

404
